# Fábrica de Treinamento DATALUS: Escala Massiva e Multi-GPU

Este notebook demonstra a **Fábrica de Treinamento DATALUS**, projetada para ambientes de computação intensiva (ex: Kaggle 2x T4 ou instâncias AWS G4).
Ao contrário do POC, focamos estritamente na ingestão massiva de dados, processamento seguro contra OOM (Out-Of-Memory) via Polars (Ingestão Preguiçosa),
Treinamento Distribuído Multi-GPU usando `torch.nn.DataParallel` e exportação para Edge via Quantização Pós-Treinamento (PTQ).

O objetivo é demonstrar a escalabilidade do framework ao lidar com milhões de registros do Sistema de Informações Hospitalares (SIH-SUS).

## 1. Configuração do Ambiente
Detectando dinamicamente o ambiente (Kaggle, Colab, Local) e instalando os pacotes necessários.

In [ ]:
import os
import sys
from pathlib import Path

def get_working_dir():
    if 'google.colab' in sys.modules:
        return '/content/datalus_factory'
    if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
        return '/kaggle/working/datalus_factory'
    return './datalus_factory'

WORKING_DIR = get_working_dir()
os.makedirs(WORKING_DIR, exist_ok=True)
print(f'Working directory: {WORKING_DIR}')

# Install dependencies if in Colab/Kaggle
if 'google.colab' in sys.modules or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    !git clone https://github.com/emanuellcs/datalus.git || true
    %cd datalus
    !python -m pip install -e '.[dev]' pysus polars
    sys.path.append(os.getcwd())


## 2. Ingestão Massiva de Dados (SIH-SUS 2023 Ano Completo)
Simulamos a escala do mundo real baixando um ano inteiro de dados do SIH-SUS (São Paulo, 2023) usando a API `pysus`.
Salvar os blocos concatenados em um único arquivo Parquet massivo permite que o motor Polars do DATALUS utilize a **avaliação preguiçosa** (`pl.scan_parquet`).

Esta escolha arquitetônica garante que evitemos o esgotamento da VRAM e RAM mesmo ao processar conjuntos de dados que excedem a memória disponível, pois os dados são transmitidos de forma eficiente para a camada de codificação de tensores.

In [ ]:
from pysus.api import sih
import polars as pl

print('Downloading 12 months of SIH-SUS data for SP (2023)...')
# Scalable download and concatenation
dfs = []
for m in range(1, 13):
    print(f'Fetching month {m}...')
    df_m = sih(state='SP', year=2023, month=[m])
    dfs.append(pl.from_pandas(df_m))

df_massive = pl.concat(dfs, how='vertical_relaxed')
df_massive = df_massive.with_columns(
    pl.col('MORTE').cast(pl.Int64).fill_null(0)
)

cols_to_keep = [c for c in df_massive.columns if not c.startswith(('N_AIH', 'IDENT'))]
df_massive = df_massive.select(cols_to_keep)

raw_path = f'{WORKING_DIR}/massive_raw_sih.parquet'
df_massive.write_parquet(raw_path)
print(f'Massive dataset saved to {raw_path} | Shape: {df_massive.shape}')

## 3. Ingestão de Alto Desempenho
O comando `ingest` mapeia o Parquet massivo bruto para o formato pronto para tensores, lidando com frequências categóricas e transformações de quantis numéricos.

In [ ]:
!datalus ingest {WORKING_DIR}/massive_raw_sih.parquet {WORKING_DIR}/processed.parquet --schema-path {WORKING_DIR}/schema_config.json --target-column MORTE

## 4. Fábrica de Treinamento Distribuído Multi-GPU
Esta é a fase central de computação. Passamos `--gpu "0,1"` para distribuir a carga entre múltiplas GPUs.
O DATALUS mascara automaticamente as variáveis de ambiente e encapsula o denoiser MLP residual em `torch.nn.DataParallel` para um rendimento escalável.

**Componentes Chave:**
- **Escalonamento de Cosseno:** Controle sofisticado de variância de difusão.
- **Rastreamento EMA:** Média Móvel Exponencial dos pesos para garantir estabilidade estrutural.
- **Precisão Mista (AMP):** Treinamento mais rápido com menor pegada de VRAM.
- **Checkpoint Determinístico:** Estados de treinamento retomáveis em todo o cluster distribuído.

In [ ]:
!datalus train {WORKING_DIR}/schema_config.json {WORKING_DIR}/processed.parquet {WORKING_DIR}/artifacts --epochs 50 --batch-size 2048 --checkpoint-every-steps 1000 --gpu "0,1"

## 5. Desacoplamento de Artefatos: Exportação Edge
Após o treinamento em hardware de alto desempenho, desacoplamos o modelo generativo.
Usando Quantização Pós-Treinamento (**INT8 PTQ**), comprimimos o modelo de difusão massivo float32.

Isso permite que o modelo, apesar de ter sido treinado em 2x GPUs T4, seja executado suavemente em CPUs locais ou diretamente no navegador via ONNX Runtime Web.

In [ ]:
!datalus export-onnx {WORKING_DIR}/artifacts/checkpoints/checkpoint_latest.pt {WORKING_DIR}/artifacts/encoder_config.json {WORKING_DIR}/artifacts --quantize